# 02 编码器模块 (core.encoders)

提供各类特征编码功能，所有编码器均遵循 sklearn Transformer 接口规范。

**支持的编码器：**
- `WOEEncoder`: 证据权重编码（风控专用）
- `TargetEncoder`: 目标编码
- `CountEncoder`: 计数编码
- `OneHotEncoder`: 独热编码
- `OrdinalEncoder`: 序数编码
- `QuantileEncoder`: 分位数编码
- `CatBoostEncoder`: CatBoost编码（有序目标编码）
- `GBMEncoder`: 梯度提升树编码器（叶子节点/Embedding）
- `CardinalityEncoder`: 高基数降维编码器

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [ ]:
import os, sys
sys.path.append('../')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from hscredit import init_setting
from hscredit.core.encoders import (
    WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder,
    OrdinalEncoder, QuantileEncoder, CatBoostEncoder,
    GBMEncoder, CardinalityEncoder
)
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import ks, auc

init_setting()

df = pd.read_excel('hscredit_yyp.xlsx')
df['target'] = (df['MOB1'] > 3).astype(int)

numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '轻花老客海纳子分V1']

# 先分箱，再进行编码
X_raw = df[numeric_features].fillna(df[numeric_features].median())

binner = OptimalBinning(method='quantile', max_n_bins=5, min_bin_size=0.05)
df_binned = binner.fit_transform(X_raw, df['target']).astype(str)
df_binned.columns = [f'{c}_bin' for c in df_binned.columns]

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")
print(f"分箱后列: {df_binned.columns.tolist()}")
display(df_binned.head())

## 1. WOEEncoder - 证据权重编码（风控专用）

In [ ]:
# WOE编码
woe_encoder = WOEEncoder()
woe_encoder.fit(df_binned, df['target'])
df_woe = woe_encoder.transform(df_binned.copy())

print("WOE编码结果预览:")
display(df_woe.head())

print("\nWOE映射表:")
for col in df_woe.columns:
    print(f"\n{col}:")
    mapping = woe_encoder.get_mapping(col)
    for k, v in mapping.items():
        print(f"  {k}: {v:.4f}")

## 2. TargetEncoder - 目标编码

In [ ]:
# Target编码
target_encoder = TargetEncoder(
    cols=['score_c3_bin'],
    smoothing=10
)
target_encoder.fit(df_binned[['score_c3_bin']], df['target'])

df_target = df_binned[['score_c3_bin']].copy()
df_target['target_encoded'] = target_encoder.transform(df_target)['score_c3_bin']

print("Target编码结果预览:")
display(df_target.head(10))

## 3. CountEncoder - 计数编码

In [ ]:
# Count编码
count_encoder = CountEncoder(
    cols=['score_c3_bin'],
    normalize=False
)
count_encoder.fit(df_binned[['score_c3_bin']])

df_count = df_binned[['score_c3_bin']].copy()
df_count['count_encoded'] = count_encoder.transform(df_count)['score_c3_bin']

print("Count编码结果预览:")
display(df_count.head(10))

## 4. OneHotEncoder - 独热编码

In [ ]:
# OneHot编码
onehot_encoder = OneHotEncoder(
    cols=['score_c3_bin'],
    drop='first'
)
onehot_encoder.fit(df_binned[['score_c3_bin']], df['target'])

df_onehot = onehot_encoder.transform(df_binned[['score_c3_bin']].copy())

print(f"OneHot编码后特征数: {df_onehot.shape[1]}")
print("OneHot编码结果预览:")
display(df_onehot.head(10))

## 5. OrdinalEncoder - 序数编码

In [ ]:
# Ordinal编码 - 按指定顺序映射
ordinal_encoder = OrdinalEncoder(
    cols=['score_c3_bin'],
    mapping={'score_c3_bin': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}}
)
ordinal_encoder.fit(df_binned[['score_c3_bin']])

df_ordinal = df_binned[['score_c3_bin']].copy()
df_ordinal['ordinal_encoded'] = ordinal_encoder.transform(df_ordinal)['score_c3_bin']

print("Ordinal编码结果预览:")
display(df_ordinal.head(10))

## 6. QuantileEncoder - 分位数编码

In [ ]:
# Quantile编码 - 按分位数排名编码
quantile_encoder = QuantileEncoder(
    cols=['score_c3_bin'],
    n_quantiles=5
)
quantile_encoder.fit(df_binned[['score_c3_bin']], df['target'])

df_quantile = df_binned[['score_c3_bin']].copy()
df_quantile['quantile_encoded'] = quantile_encoder.transform(df_quantile)['score_c3_bin']

print("Quantile编码结果预览:")
display(df_quantile.head(10))

## 7. CatBoostEncoder - CatBoost有序目标编码

In [ ]:
# CatBoost编码 - 有序目标编码，减少过拟合
catboost_encoder = CatBoostEncoder(
    cols=['score_c3_bin'],
    smoothing=10,
    random_state=42
)
catboost_encoder.fit(df_binned[['score_c3_bin']], df['target'])

df_catboost = df_binned[['score_c3_bin']].copy()
df_catboost['catboost_encoded'] = catboost_encoder.transform(df_catboost)['score_c3_bin']

print("CatBoost编码结果预览:")
display(df_catboost.head(10))

## 8. GBMEncoder - 梯度提升树编码器

In [ ]:
# GBM编码 - 使用梯度提升树提取叶子节点或Embedding
X_for_gbm = df[numeric_features].fillna(df[numeric_features].median())
y_for_gbm = df['target']

X_train_gbm, X_test_gbm, y_train_gbm, y_test_gbm = train_test_split(
    X_for_gbm, y_for_gbm, test_size=0.3, random_state=42
)

# 叶子节点编码
gbm_leaves = GBMEncoder(
    model_type='lightgbm',
    n_estimators=50,
    max_depth=3,
    output='leaves',
    random_state=42
)
gbm_leaves.fit(X_train_gbm, y_train_gbm)
leaves_train = gbm_leaves.transform(X_train_gbm)

print("GBM叶子节点编码结果预览:")
print(f"形状: {leaves_train.shape}")
display(pd.DataFrame(leaves_train[:5], columns=[f'leaf_{i}' for i in range(leaves_train.shape[1])]))

## 9. CardinalityEncoder - 高基数降维编码器

In [ ]:
# Cardinality编码 - 高基数类别降维
# 先创建一些高基数类别特征
np.random.seed(42)
df['category_1'] = pd.cut(df['中智小牛分C3'].fillna(df['中智小牛分C3'].median()), bins=20).astype(str)
df['category_2'] = pd.cut(df['珊瑚92'].fillna(df['珊瑚92'].median()), bins=15).astype(str)

card_encoder = CardinalityEncoder(
    cols=['category_1', 'category_2'],
    max_categories=5,
)
card_encoder.fit(df[['category_1', 'category_2']])

df_card = card_encoder.transform(df[['category_1', 'category_2']].copy())

print("Cardinality编码结果预览:")
display(df_card.head(10))

print("\n各类别的频次:")
for col in df_card.columns:
    print(f"\n{col}:")
    print(df_card[col].value_counts().head())

## 10. Pipeline中使用编码器

In [ ]:
from sklearn.pipeline import Pipeline
from hscredit.core.models import LogisticRegression

# 准备数据
feature_cols = ['score_c3_bin', 'score_coral_bin', 'score_fraud_bin', 'score_haina_bin']
X = df_binned[feature_cols].copy()
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 对比不同编码器在LR模型上的效果
encoders = {
    'WOE': WOEEncoder(),
    'Target': TargetEncoder(cols=feature_cols, smoothing=10),
    'Count': CountEncoder(cols=feature_cols),
}

results = []
for name, encoder in encoders.items():
    X_train_enc = encoder.fit_transform(X_train, y_train)
    X_test_enc = encoder.transform(X_test)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_enc, y_train)
    y_prob = model.predict_proba(X_test_enc)[:, 1]
    
    results.append({
        '编码器': name,
        'KS': ks(y_test, y_prob),
        'AUC': auc(y_test, y_prob),
    })

print("=== 编码器效果对比 ===")
display(pd.DataFrame(results))